# Group 20B - Mobile Payment Fraud Detection - Cleaned Notebook v3

This is a separate cleaned copy of the original AI4ALL Group 20B PaySim fraud-detection notebook. The original notebook and the v1 scaffold are not modified.

Beginner note: this version is organized so each section creates the variables needed by later sections. That makes the notebook safer to run from top to bottom in a fresh runtime.


## What Changed in This Cleaned Version

- `evaluate_model` is defined before any model calls it.
- The train/validation/test split is created before model training.
- Safe engineered features from `df_fe` are used for modeling.
- `nameOrig`, `nameDest`, `isFlaggedFraud`, `orig_txn_count`, and `dest_txn_count` are excluded from predictors.
- `isFlaggedFraud` is kept as a baseline and evaluated on the same untouched test set.
- Logistic Regression and Random Forest keep the teammates' original modeling ideas, with only the necessary fixes for clean data preparation, validation thresholds, and shared metrics.
- XGBoost uses the same leakage-safe predictors and validation-only threshold selection as the other models.


## 1. Setup and Imports

This section imports all packages before any data work starts.


In [2]:
import pandas as pd
import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed in this runtime; skipping plotting support.")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    average_precision_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


## 2. Development Settings

`USE_SMALL_SAMPLE` is intentionally set to `True`. This lets us test the notebook on a small stratified sample first.

`DEVELOPMENT_MODE` creates a tiny PaySim-shaped dataset only when the real Drive file is not available. In Colab with the real dataset path, set `DEVELOPMENT_MODE = False` and keep `USE_SMALL_SAMPLE = True` for the first project run.


In [3]:
RANDOM_STATE = 42
USE_SMALL_SAMPLE = True
SAMPLE_SIZE = 100_000
DEVELOPMENT_MODE = False

# Expected mounted Google Drive path in Colab.
# If your file is stored in a different Drive folder, update only this value.
DATA_PATH = 'data/PaySim_DS.csv'


## 3. Load Dataset

This keeps the original Drive-based loading approach. The fallback development data is only for checking that the notebook runs from top to bottom when the real CSV is not available.


In [4]:
def make_development_paysim_sample(n_rows=1_000, fraud_rate=0.04, random_state=RANDOM_STATE):
    """Create a small PaySim-shaped dataset for local run-order testing only."""
    rng = np.random.default_rng(random_state)
    n_fraud = max(2, int(n_rows * fraud_rate))
    n_normal = n_rows - n_fraud

    is_fraud = np.array([1] * n_fraud + [0] * n_normal)
    rng.shuffle(is_fraud)

    txn_type = np.where(
        is_fraud == 1,
        rng.choice(['TRANSFER', 'CASH_OUT'], size=n_rows),
        rng.choice(['PAYMENT', 'CASH_OUT', 'TRANSFER', 'DEBIT', 'CASH_IN'], size=n_rows),
    )

    amount = rng.lognormal(mean=10.5, sigma=1.0, size=n_rows).round(2)
    oldbalance_org = rng.lognormal(mean=11.0, sigma=1.2, size=n_rows).round(2)
    newbalance_orig = np.maximum(oldbalance_org - amount, 0).round(2)

    oldbalance_dest = rng.lognormal(mean=10.8, sigma=1.3, size=n_rows).round(2)
    newbalance_dest = (oldbalance_dest + np.where(txn_type == 'PAYMENT', 0, amount)).round(2)

    fraud_idx = is_fraud == 1
    newbalance_orig[fraud_idx] = 0
    amount[fraud_idx] = np.maximum(amount[fraud_idx], oldbalance_org[fraud_idx]).round(2)

    is_flagged = ((txn_type == 'TRANSFER') & (amount > 200_000) & (is_fraud == 1)).astype(int)

    return pd.DataFrame({
        'step': rng.integers(1, 744, size=n_rows),
        'type': txn_type,
        'amount': amount,
        'nameOrig': [f'C{i:010d}' for i in range(n_rows)],
        'oldbalanceOrg': oldbalance_org,
        'newbalanceOrig': newbalance_orig,
        'nameDest': [f'M{i:010d}' for i in range(n_rows)],
        'oldbalanceDest': oldbalance_dest,
        'newbalanceDest': newbalance_dest,
        'isFraud': is_fraud,
        'isFlaggedFraud': is_flagged,
    })


if IN_COLAB and not DEVELOPMENT_MODE:
    drive.mount('/content/drive/')

    import os

    MY_DRIVE_ROOT = '/content/drive/MyDrive'
    LIKELY_DATA_PATHS = [
        DATA_PATH,
        '/content/drive/MyDrive/PaySim_DS.csv',
        '/content/drive/MyDrive/PaySim_DS',
    ]

    matching_paths = []

    for possible_path in LIKELY_DATA_PATHS:
        if os.path.exists(possible_path):
            matching_paths.append(possible_path)

    if not matching_paths:
        for root, dirs, files in os.walk(MY_DRIVE_ROOT):
            for filename in files:
                if filename.startswith('PaySim_DS'):
                    matching_paths.append(os.path.join(root, filename))

    matching_paths = sorted(set(matching_paths))

    if len(matching_paths) == 1:
        DATA_PATH = matching_paths[0]
        print('Resolved DATA_PATH:')
        print(DATA_PATH)
        df = pd.read_csv(DATA_PATH)
    elif len(matching_paths) > 1:
        print('Multiple PaySim_DS files were found. Please choose one and set DATA_PATH to that exact path:')
        for path in matching_paths:
            print(path)
        raise ValueError('Multiple PaySim_DS files found. Choose one path and update DATA_PATH.')
    else:
        print('No PaySim_DS file was found under the likely paths or recursive MyDrive search.')
        print('Expected likely paths checked:')
        for path in LIKELY_DATA_PATHS:
            print(path)
        print('Top-level contents of /content/drive/MyDrive:')
        if os.path.exists(MY_DRIVE_ROOT):
            for item in sorted(os.listdir(MY_DRIVE_ROOT)):
                print(os.path.join(MY_DRIVE_ROOT, item))
        else:
            print('/content/drive/MyDrive does not exist after mounting Drive.')
        raise FileNotFoundError('No file whose name begins with PaySim_DS was found under /content/drive/MyDrive.')
else:
    DEVELOPMENT_MODE = True
    df = make_development_paysim_sample()
    print('Using tiny development data because the real Colab Drive dataset is not available here.')

df.shape


Using tiny development data because the real Colab Drive dataset is not available here.


(1000, 11)

## 4. Quick Data Checks and EDA

These checks confirm the dataset loaded correctly before we build features or train models.


In [5]:
df.head()


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,88,PAYMENT,8167.30,C0000000000,339921.65,331754.35,M0000000000,34331.23,34331.23,0,0
1,641,PAYMENT,38309.68,C0000000001,49627.19,11317.51,M0000000001,7183.20,7183.20,0,0
2,375,CASH_IN,32235.74,C0000000002,428662.39,396426.65,M0000000002,311770.84,344006.58,0,0
3,358,PAYMENT,116954.25,C0000000003,32326.30,0.00,M0000000003,29408.27,29408.27,0,0
4,686,CASH_OUT,30374.08,C0000000004,84918.41,54544.33,M0000000004,289995.88,320369.96,0,0


In [ ]:
df.info()


In [6]:
print("Fraud counts:")
print(df["isFraud"].value_counts())

print("\nTransaction type counts:")
print(df["type"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())


Fraud counts:
isFraud
0    960
1     40
Name: count, dtype: int64

Transaction type counts:
type
TRANSFER    206
CASH_IN     205
CASH_OUT    205
PAYMENT     201
DEBIT       183
Name: count, dtype: int64

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


## 5. Safe Feature Engineering

This section uses the safe row-level engineered features from `df_fe`.

Important leakage note: `orig_txn_count` and `dest_txn_count` are created for EDA only in this version. They are excluded from predictors because account-frequency features calculated before the split can leak information from validation/test rows into training.


In [ ]:
df_fe = df.copy()

# Log transform amount to reduce the effect of very large transactions.
df_fe["amount_log"] = np.log1p(df_fe["amount"])

# Balance error features compare the transaction amount with balance changes.
df_fe["orig_balance_error"] = (
    df_fe["oldbalanceOrg"] - df_fe["newbalanceOrig"] - df_fe["amount"]
).abs()

df_fe["dest_balance_error"] = (
    df_fe["newbalanceDest"] - df_fe["oldbalanceDest"] - df_fe["amount"]
).abs()

# Indicator for sender account being emptied.
df_fe["orig_account_emptied"] = (
    (df_fe["oldbalanceOrg"] > 0) &
    (df_fe["newbalanceOrig"] == 0)
).astype(int)

# Time-based features from simulation step.
df_fe["hour_of_day"] = df_fe["step"] % 24
df_fe["day"] = df_fe["step"] // 24

# EDA only in this cleaned version. These are not used as predictors.
df_fe["orig_txn_count"] = df_fe.groupby("nameOrig")["step"].transform("count")
df_fe["dest_txn_count"] = df_fe.groupby("nameDest")["step"].transform("count")

df_fe.head()


## 6. Feature Selection and Baseline Separation

`isFraud` is the target we want to predict. `isFlaggedFraud` is not used as a model input; it is kept separately as the original baseline for comparison.

We also exclude raw account IDs and account-frequency features in this first cleaned implementation.


In [ ]:
TARGET_COLUMN = "isFraud"
BASELINE_COLUMN = "isFlaggedFraud"

EXCLUDED_PREDICTOR_COLUMNS = [
    TARGET_COLUMN,
    BASELINE_COLUMN,
    "nameOrig",
    "nameDest",
    "orig_txn_count",
    "dest_txn_count",
]

X = df_fe.drop(columns=EXCLUDED_PREDICTOR_COLUMNS)
y = df_fe[TARGET_COLUMN]
baseline_flag = df_fe[BASELINE_COLUMN]

print("Predictor columns:")
print(X.columns.tolist())


## 7. Small Stratified Sample for First Testing

This notebook should be tested on a small stratified sample first. Do not switch `USE_SMALL_SAMPLE` to `False` for a full 6.3-million-row run without project-owner approval.


In [ ]:
if USE_SMALL_SAMPLE:
    sample_size = min(SAMPLE_SIZE, len(df_fe))

    if sample_size == len(df_fe):
        sample_index = df_fe.index
    else:
        sample_index, _ = train_test_split(
            df_fe.index,
            train_size=sample_size,
            random_state=RANDOM_STATE,
            shuffle=True,
            stratify=y,
        )

    X_model = X.loc[sample_index].copy()
    y_model = y.loc[sample_index].copy()
    baseline_model = baseline_flag.loc[sample_index].copy()
else:
    raise RuntimeError(
        "Full-dataset training is intentionally blocked in this cleaned version. "
        "Ask the project owner before running all 6.3 million rows."
    )

print("Rows used:", len(X_model))
print("Fraud rate:")
print(y_model.value_counts(normalize=True) * 100)


## 8. Train, Validation, and Test Split

The validation set is used to choose a model threshold. The test set is untouched until final evaluation.


In [ ]:
X_train, X_temp, y_train, y_temp, baseline_train, baseline_temp = train_test_split(
    X_model,
    y_model,
    baseline_model,
    train_size=0.70,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=y_model,
)

X_val, X_test, y_val, y_test, baseline_val, baseline_test = train_test_split(
    X_temp,
    y_temp,
    baseline_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=y_temp,
)

print("Training set:")
print(y_train.value_counts(), end="\n\n")
print("Validation set:")
print(y_val.value_counts(), end="\n\n")
print("Test set:")
print(y_test.value_counts())


## 9. Shared Preprocessing Helpers

Logistic Regression needs scaled numeric features. Random Forest does not need scaling. Both models use the same cleaned feature set created above.


In [ ]:
def prepare_numeric_features(X_train_data, X_other_data):
    """Keep numeric columns and fill missing values using training-set means."""
    numeric_cols = X_train_data.select_dtypes(include="number").columns.tolist()
    X_train_numeric = X_train_data[numeric_cols].copy()
    X_other_numeric = X_other_data[numeric_cols].copy()

    for col in numeric_cols:
        mean_val = X_train_numeric[col].mean()
        X_train_numeric[col] = X_train_numeric[col].fillna(mean_val)
        X_other_numeric[col] = X_other_numeric[col].fillna(mean_val)

    return X_train_numeric, X_other_numeric


def fit_tree_preprocessor(X_train_data):
    """One-hot encode training data and remember the final training columns."""
    X_train_tree = pd.get_dummies(X_train_data.copy(), columns=["type"], drop_first=False)

    for col in X_train_tree.columns:
        X_train_tree[col] = X_train_tree[col].fillna(X_train_tree[col].mean())

    return X_train_tree, X_train_tree.columns.tolist(), X_train_tree.mean(numeric_only=True)


def transform_tree_features(X_other_data, training_columns, training_means):
    """Apply the training one-hot columns and training means to validation/test data."""
    X_other_tree = pd.get_dummies(X_other_data.copy(), columns=["type"], drop_first=False)
    X_other_tree = X_other_tree.reindex(columns=training_columns, fill_value=0)

    for col in training_columns:
        X_other_tree[col] = X_other_tree[col].fillna(training_means[col])

    return X_other_tree


## 10. Shared Evaluation and Threshold Helpers

`evaluate_model` is defined here before any model calls it.

Threshold rule: validation data is used to pick a model threshold. The test set is only used after the threshold has already been chosen.


In [ ]:
DEFAULT_THRESHOLD = 0.50
THRESHOLD_GRID = np.round(np.arange(0.05, 1.00, 0.05), 2)


def make_fraud_flags(y_score, threshold=DEFAULT_THRESHOLD):
    return (np.asarray(y_score) >= threshold).astype(int)


def select_threshold_from_validation(y_true, y_score, thresholds=THRESHOLD_GRID):
    """Choose the threshold with the best validation F1 score."""
    rows = []
    for threshold in thresholds:
        y_pred = make_fraud_flags(y_score, threshold)
        rows.append({
            "Threshold": threshold,
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "Number Flagged": int(y_pred.sum()),
            "Percentage Flagged": float(y_pred.mean() * 100),
        })

    threshold_df = pd.DataFrame(rows)
    best_row = threshold_df.sort_values(
        by=["F1", "Recall", "Precision"],
        ascending=False,
    ).iloc[0]

    return float(best_row["Threshold"]), threshold_df


def evaluate_model(model_name, y_true, y_pred, y_score=None, threshold=None):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    number_flagged = int(np.sum(y_pred))
    percentage_flagged = float(np.mean(y_pred) * 100)

    results = {
        "Model": model_name,
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "False Positive Rate": false_positive_rate,
        "Number Flagged": number_flagged,
        "Percentage Flagged": percentage_flagged,
        "Confusion Matrix": cm,
        "Classification Report": classification_report(y_true, y_pred, zero_division=0),
    }

    if y_score is not None:
        results["PR-AUC / Average Precision"] = average_precision_score(y_true, y_score)
    else:
        results["PR-AUC / Average Precision"] = np.nan

    print(model_name)
    print("Threshold:", threshold)
    print("Confusion matrix:")
    print(cm)
    print("\nClassification report:")
    print(results["Classification Report"])

    return results


## 11. Logistic Regression

### Changes made to the original Logistic Regression section

- Kept the original idea of using numeric columns only, `StandardScaler`, and `LogisticRegression`.
- Moved evaluation after the shared `evaluate_model` helper so the notebook can run top to bottom.
- Used the cleaned feature set, so `isFlaggedFraud`, account IDs, and account-frequency features are not predictors.
- Used train-set means to fill missing values, then applied those same means to validation/test data.
- Added validation-based threshold selection without using the test set for tuning.


In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd  # Required for select_dtypes and fillna


def train_and_evaluate_logistic_regression(
    X_train_data,
    y_train_data,
    X_val_data,
    y_val_data,
    X_test_data,
    y_test_data,
):
    # Keep only numeric columns for modeling
    # (the cleaned X data may still include non-numeric columns like 'type')
    X_train_numeric, X_val_numeric = prepare_numeric_features(X_train_data, X_val_data)
    _, X_test_numeric = prepare_numeric_features(X_train_data, X_test_data)

    print(f"Training Logistic Regression model using {X_train_numeric.shape[1]} numeric features.")
    # print("Features used:", X_train_numeric.columns.tolist()) # Uncomment to see the list of features

    model_pipeline = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')  # Added solver for convergence
    )

    # Train the model
    model_pipeline.fit(X_train_numeric, y_train_data)

    # Use validation data only to choose a threshold
    y_val_proba = model_pipeline.predict_proba(X_val_numeric)[:, 1]
    selected_threshold, lr_threshold_table = select_threshold_from_validation(y_val_data, y_val_proba)

    # Make predictions on the test set using both default and validation-selected thresholds
    y_test_proba = model_pipeline.predict_proba(X_test_numeric)[:, 1]
    y_pred_default = make_fraud_flags(y_test_proba, DEFAULT_THRESHOLD)
    y_pred_selected = make_fraud_flags(y_test_proba, selected_threshold)

    default_results = evaluate_model(
        "Logistic Regression (threshold 0.50)",
        y_test_data,
        y_pred_default,
        y_test_proba,
        threshold=DEFAULT_THRESHOLD,
    )
    selected_results = evaluate_model(
        "Logistic Regression (validation-selected threshold)",
        y_test_data,
        y_pred_selected,
        y_test_proba,
        threshold=selected_threshold,
    )

    return {
        "model_object": model_pipeline,
        "feature_names": X_train_numeric.columns.tolist(),
        "threshold_table": lr_threshold_table,
        "default_results": default_results,
        "selected_results": selected_results,
        "test_scores": y_test_proba,
        "test_flags_default": y_pred_default,
        "test_flags_selected": y_pred_selected,
    }


lr_output = train_and_evaluate_logistic_regression(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
)


## 12. Random Forest

### Changes made to the original Random Forest section

- Kept the original idea of dropping identifier columns, one-hot encoding `type`, aligning train/test columns, mean-imputing missing values, and using `class_weight='balanced'`.
- The cleaned feature set already excludes `nameOrig`, `nameDest`, `isFlaggedFraud`, `orig_txn_count`, and `dest_txn_count`; the defensive drop step remains so the intent is clear.
- Used validation data only to choose a threshold.
- Used the untouched test set only for final evaluation.


In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier
import pandas as pd


def train_and_evaluate_random_forest(
    X_train_data,
    y_train_data,
    X_val_data,
    y_val_data,
    X_test_data,
    y_test_data,
):
    X_train_rf = X_train_data.copy()
    X_val_rf = X_val_data.copy()
    X_test_rf = X_test_data.copy()

    # Drop ID columns - high-cardinality identifiers, not useful as direct predictors
    drop_cols = ['nameOrig', 'nameDest', 'isFlaggedFraud', 'orig_txn_count', 'dest_txn_count']
    X_train_rf = X_train_rf.drop(columns=[c for c in drop_cols if c in X_train_rf.columns])
    X_val_rf = X_val_rf.drop(columns=[c for c in drop_cols if c in X_val_rf.columns])
    X_test_rf = X_test_rf.drop(columns=[c for c in drop_cols if c in X_test_rf.columns])

    # One-hot encode 'type' - RF can use it directly and it's a strong fraud signal
    # (PaySim fraud only ever occurs in TRANSFER and CASH_OUT transactions)
    X_train_rf, rf_training_columns, rf_training_means = fit_tree_preprocessor(X_train_rf)
    X_val_rf = transform_tree_features(X_val_rf, rf_training_columns, rf_training_means)
    X_test_rf = transform_tree_features(X_test_rf, rf_training_columns, rf_training_means)

    print(f"Training Random Forest using {X_train_rf.shape[1]} features.")

    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=5,
        class_weight='balanced',   # addresses class imbalance noted in Sources of Bias
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train_rf, y_train_data)

    # Use validation data only to choose a threshold
    y_val_proba = rf_model.predict_proba(X_val_rf)[:, 1]
    selected_threshold, rf_threshold_table = select_threshold_from_validation(y_val_data, y_val_proba)

    # Make predictions on the test set using both default and validation-selected thresholds
    y_test_proba = rf_model.predict_proba(X_test_rf)[:, 1]
    y_pred_default = make_fraud_flags(y_test_proba, DEFAULT_THRESHOLD)
    y_pred_selected = make_fraud_flags(y_test_proba, selected_threshold)

    default_results = evaluate_model(
        "Random Forest (threshold 0.50)",
        y_test_data,
        y_pred_default,
        y_test_proba,
        threshold=DEFAULT_THRESHOLD,
    )
    selected_results = evaluate_model(
        "Random Forest (validation-selected threshold)",
        y_test_data,
        y_pred_selected,
        y_test_proba,
        threshold=selected_threshold,
    )

    return {
        "model_object": rf_model,
        "feature_names": X_train_rf.columns.tolist(),
        "threshold_table": rf_threshold_table,
        "default_results": default_results,
        "selected_results": selected_results,
        "test_scores": y_test_proba,
        "test_flags_default": y_pred_default,
        "test_flags_selected": y_pred_selected,
    }


rf_output = train_and_evaluate_random_forest(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
)


## 13. XGBoost

XGBoost uses leakage-safe predictors. The imbalance weight and decision threshold are calculated from the training and validation splits only.


In [15]:
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "Install project dependencies with `pip install -r requirements.txt`, "
        "then restart the notebook kernel."
    ) from exc

# Fit preprocessing only on training data and align validation/test features.
# This avoids categorical-dtype compatibility errors across XGBoost versions.
X_train_xgb, xgb_columns, xgb_means = fit_tree_preprocessor(X_train)
X_val_xgb = transform_tree_features(X_val, xgb_columns, xgb_means)
X_test_xgb = transform_tree_features(X_test, xgb_columns, xgb_means)

# 2. Calculate the exact scale_pos_weight from the training data to handle class imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
if pos_count == 0 or neg_count == 0:
    raise ValueError("XGBoost training data must contain both target classes.")
imbalance_ratio = neg_count / pos_count
print(f"Calculated scale_pos_weight: {imbalance_ratio:.2f}\n")

# 3. Initialize the XGBoost Classifier
xgb_model = XGBClassifier(
    scale_pos_weight=imbalance_ratio,
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='aucpr',
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)

# 4. Train only on the training split.
print("Training XGBoost on training set...")
xgb_model.fit(
    X_train_xgb, y_train
)
print("Training complete.\n")

# 5. Predict on the hidden Test Set
xgb_val_scores = xgb_model.predict_proba(X_val_xgb)[:, 1]
xgb_selected_threshold, xgb_threshold_table = select_threshold_from_validation(
    y_val, xgb_val_scores
)
xgb_test_scores = xgb_model.predict_proba(X_test_xgb)[:, 1]
y_pred_xgb = make_fraud_flags(xgb_test_scores, xgb_selected_threshold)

# 6. Output text performance metrics for the evaluation table
print("=== XGBoost Final Test Results ===")
print("\nClassification Report:")
print(f"Validation-selected threshold: {xgb_selected_threshold:.2f}")
print(classification_report(y_test, y_pred_xgb, zero_division=0))

xgb_precision = precision_score(y_test, y_pred_xgb, zero_division=0)
xgb_recall = recall_score(y_test, y_pred_xgb, zero_division=0)
xgb_f1 = f1_score(y_test, y_pred_xgb, zero_division=0)

print(f"Final Precision: {xgb_precision:.4f}")
print(f"Final Recall: {xgb_recall:.4f}")
print(f"Final F1-Score: {xgb_f1:.4f}\n")

# XGBoost results are also available to the shared notebook tables below.
xgb_output = {
    "model_object": xgb_model,
    "feature_names": xgb_columns,
    "scale_pos_weight": imbalance_ratio,
    "threshold_table": xgb_threshold_table,
    "test_scores": xgb_test_scores,
    "test_flags_selected": y_pred_xgb,
}


ImportError: Install project dependencies with `pip install -r requirements.txt`, then restart the notebook kernel.

## 14. Threshold-Based Fraud Flagging on the Test Set

This creates `modelFlaggedFraud` columns for the test set. The selected threshold was chosen from validation data only.


In [ ]:
test_results_df = X_test.copy()
test_results_df["isFraud"] = y_test.values
test_results_df["isFlaggedFraud"] = baseline_test.values

test_results_df["lr_modelFlaggedFraud_default_050"] = lr_output["test_flags_default"]
test_results_df["lr_modelFlaggedFraud_validation_selected"] = lr_output["test_flags_selected"]
test_results_df["rf_modelFlaggedFraud_default_050"] = rf_output["test_flags_default"]
test_results_df["rf_modelFlaggedFraud_validation_selected"] = rf_output["test_flags_selected"]

# Generic modelFlaggedFraud column for the current best available model in this version.
# This uses Random Forest with the validation-selected threshold because Gradient Boosting is not implemented yet.
test_results_df["modelFlaggedFraud"] = test_results_df["rf_modelFlaggedFraud_validation_selected"]

test_results_df.head()


## 15. Original `isFlaggedFraud` Baseline Evaluation

This compares the model-created fraud flag with the original PaySim `isFlaggedFraud` rule using the same untouched test set.


In [ ]:
baseline_results = evaluate_model(
    "Original isFlaggedFraud Baseline",
    y_test,
    baseline_test,
    y_score=None,
    threshold="PaySim rule",
)


## 16. Model-Comparison DataFrame

This table compares the baseline, default-threshold models, and validation-selected-threshold models in one place.


In [ ]:
model_results = [
    baseline_results,
    lr_output["default_results"],
    lr_output["selected_results"],
    rf_output["default_results"],
    rf_output["selected_results"],
]

comparison_columns = [
    "Model",
    "Threshold",
    "Precision",
    "Recall",
    "F1",
    "PR-AUC / Average Precision",
    "False Positive Rate",
    "Number Flagged",
    "Percentage Flagged",
]

model_comparison_df = pd.DataFrame(model_results)
model_comparison_df = model_comparison_df.reindex(columns=comparison_columns)
model_comparison_df


NameError: name 'baseline_results' is not defined

## 17. Key Findings and Next Steps

These findings are based on the saved outputs from the real PaySim development sample run. The test set had 15,000 transactions, including 19 fraud cases.

1. Random Forest performed best on this development sample. At both the default threshold of 0.50 and the validation-selected threshold of 0.30, it found all 19 fraud cases and made no false positive fraud flags. Its precision, recall, F1, and PR-AUC / Average Precision were all 1.000000, with a false-positive rate of 0.000000. It flagged 19 transactions, or 0.126667% of the test set.

2. Logistic Regression found some fraud, but missed 8 of the 19 fraud cases at both thresholds. At the default 0.50 threshold, it had precision 0.916667, recall 0.578947, F1 0.709677, and PR-AUC / Average Precision 0.671791. It flagged 12 transactions, or 0.080000% of the test set, with a very low false-positive rate of 0.000067.

3. Lowering Logistic Regression to the validation-selected threshold of 0.15 did not improve recall on the test set. Recall stayed at 0.578947, but precision dropped from 0.916667 to 0.846154 and F1 dropped from 0.709677 to 0.687500. It flagged one extra transaction, 13 instead of 12, raising the flagged percentage from 0.080000% to 0.086667% and the false-positive rate from 0.000067 to 0.000134.

4. The original isFlaggedFraud baseline did not catch any fraud cases in this test set. It flagged 0 transactions, so its precision, recall, and F1 were all 0.000000. Its false-positive rate was also 0.000000, but only because it made no fraud flags at all.

5. The validation-selected thresholds did not change the Random Forest test results, but they did make Logistic Regression slightly less precise. For this sample, Random Forest was the strongest model because it had perfect fraud recall without increasing false positives, while Logistic Regression traded a small number of extra flags for no additional fraud cases found.

Before any further redesign, ask the project owner.
